# People on Banknotes

Whose faces appear on banknotes?

The file `people-on-banknotes.csv` contains data about individuals featured on banknotes from 38 countries. This dataset spans all 22 subregions and sub-subregions of the world, as defined by the United Nations Statistics Division's geoscheme.

It profiles 241 people, detailing their occupations and the year they first appeared on a banknote. Additionally, it includes their year of death — or `NaN` if they were still alive when the dataset was compiled.

Most banknotes were issued after the featured individual’s death. The column `first_death_diff` calculates the difference between the year of their first appearance on a banknote and their year of death (or remains `NaN` if the person was still living at the time of curation).




In [1]:
# FOR GOOGLE COLAB ONLY.
# Uncomment and run the code below. A dialog will appear to upload files.
# Upload 'people-on-banknotes.csv'.

# from google.colab import files
# uploaded = files.upload()

In [2]:
import pandas as pd

df = pd.read_csv('people-on-banknotes.csv')
df

,country,currency,name,gender,occupation,value,first_appearance,death,first_death_diff,currency_code
0,Argentina,Argentine Peso,Eva Perón,F,Activist,100,2012,1952,60.0,ARS
1,Argentina,Argentine Peso,Julio Argentino Roca,M,Head of Gov't,100,1988,1914,74.0,ARS
2,Argentina,Argentine Peso,Domingo Faustino Sarmiento,M,Head of Gov't,50,1999,1888,111.0,ARS
3,Argentina,Argentine Peso,Juan Manuel de Rosas,M,Politician,20,1992,1877,115.0,ARS
4,Argentina,Argentine Peso,Manuel Belgrano,M,Founder,10,1970,1820,150.0,ARS
...,...,...,...,...,...,...,...,...,...,...
274,Venezuela,Venezuelan Bolivar,Francisco de Miranda,M,Military,200,1968,1816,152.0,VES
275,Venezuela,Venezuelan Bolivar,Simón Rodrigues,M,Educator,20,2007,1854,153.0,VES
276,Venezuela,Venezuelan Bolivar,Ezequiel Zamora,M,Military,100,2018,1860,158.0,VES
277,Venezuela,Venezuelan Bolivar,Rafael Urdaneta,M,Head of Gov't,10,2018,1845,173.0,VES


### Quick cleaning

The same person can appear on multiple banknotes. Below we drop the `value` column and remove duplicate people.

In [3]:
df = df.drop(columns=['value'])
df = df.drop_duplicates(subset="name")
df

,country,currency,name,gender,occupation,first_appearance,death,first_death_diff,currency_code
0,Argentina,Argentine Peso,Eva Perón,F,Activist,2012,1952,60.0,ARS
1,Argentina,Argentine Peso,Julio Argentino Roca,M,Head of Gov't,1988,1914,74.0,ARS
2,Argentina,Argentine Peso,Domingo Faustino Sarmiento,M,Head of Gov't,1999,1888,111.0,ARS
3,Argentina,Argentine Peso,Juan Manuel de Rosas,M,Politician,1992,1877,115.0,ARS
4,Argentina,Argentine Peso,Manuel Belgrano,M,Founder,1970,1820,150.0,ARS
...,...,...,...,...,...,...,...,...,...
274,Venezuela,Venezuelan Bolivar,Francisco de Miranda,M,Military,1968,1816,152.0,VES
275,Venezuela,Venezuelan Bolivar,Simón Rodrigues,M,Educator,2007,1854,153.0,VES
276,Venezuela,Venezuelan Bolivar,Ezequiel Zamora,M,Military,2018,1860,158.0,VES
277,Venezuela,Venezuelan Bolivar,Rafael Urdaneta,M,Head of Gov't,2018,1845,173.0,VES


### Project Ideas

- What proportion of individuals featured are male versus female?
	- Hint: Use `value_counts(normalize=True)` to calculate percentages.

- Are writers or politicians more commonly depicted?

- What percentage of featured individuals are musicians?

- What percentage of banknotes were issued before the person’s death?
	- Hint: Look for negative values or NaN in `first_death_diff`.

- Who is the oldest historical figure in the dataset?

- Which countries feature the oldest historical figures on their banknotes?
	- Hint: Group by country and aggregate the year of death using the median. Sort the results.

- What percentage of individuals died at least 100 years before appearing on a banknote?

- Which individuals appeared on a banknote just one year after their death?


## Lösung

### Vorbereitung

Bei Queen Elizabeth II steht in einer Zeile `-` als Todesjahr. Deshalb liest pandas die ganze Spalte `death` als Text ein, auch wenn die Zeile durch das Entfernen der Duplikate schon weg ist. Für Vergleiche und den Median muss die Spalte zuerst in Zahlen umgewandelt werden.

In [4]:
import doctest

# Nicht umwandelbare Werte wie "-" werden zu NaN
df["death"] = pd.to_numeric(df["death"], errors="coerce")
df["death"].dtype

dtype('float64')

### What proportion of individuals featured are male versus female?

In [5]:
def gender_shares(data: pd.DataFrame) -> pd.Series:
    """Liefert den Anteil jedes Geschlechts in Prozent.

    >>> gender_shares(pd.DataFrame({"gender": ["M", "M", "M", "F"]})).to_dict()
    {'M': 75.0, 'F': 25.0}
    """
    return data["gender"].value_counts(normalize=True) * 100


gender_shares(df).round(2)

gender
M    78.01
F    21.99
Name: proportion, dtype: float64

**Antwort:** 78,01 % der Personen sind Männer (188) und 21,99 % sind Frauen (53).

### Are writers or politicians more commonly depicted?

In [6]:
def occupation_counts(data: pd.DataFrame, occupations: list[str]) -> pd.Series:
    """Zählt, wie viele Personen jeden der angegebenen Berufe haben.

    >>> toy = pd.DataFrame({"occupation": ["Writer", "Writer", "Politician", "STEM"]})
    >>> occupation_counts(toy, ["Writer", "Politician"]).to_dict()
    {'Writer': 2, 'Politician': 1}
    """
    return data["occupation"].value_counts().reindex(occupations, fill_value=0)


occupation_counts(df, ["Writer", "Politician", "Head of Gov't"])

occupation
Writer           45
Politician       27
Head of Gov't    38
Name: count, dtype: int64

**Antwort:** Writer (45) kommen häufiger vor als Politician (27). `Head of Gov't` (38) ist im Datensatz eine eigene Kategorie. Zählt man sie zu den Politikern, wären es 65 Personen und das Ergebnis würde sich umdrehen.

### What percentage of featured individuals are musicians?

In [7]:
def occupation_percentage(data: pd.DataFrame, occupation: str) -> float:
    """Liefert den Anteil der Personen mit dem angegebenen Beruf in Prozent.

    >>> toy = pd.DataFrame({"occupation": ["Musician", "Writer", "Writer", "Writer"]})
    >>> occupation_percentage(toy, "Musician")
    25.0
    """
    return float((data["occupation"] == occupation).mean() * 100)


round(occupation_percentage(df, "Musician"), 2)

4.98

**Antwort:** 4,98 % der Personen sind Musiker (12 von 241).

### What percentage of banknotes were issued before the person's death?

In [8]:
def issued_before_death_percentage(data: pd.DataFrame) -> float:
    """Liefert den Anteil der Zeilen in Prozent, bei denen first_death_diff negativ oder NaN ist.

    >>> toy = pd.DataFrame({"first_death_diff": [-3, None, 10, 20]})
    >>> issued_before_death_percentage(toy)
    50.0
    """
    diff = data["first_death_diff"]
    return float(((diff < 0) | diff.isna()).mean() * 100)


all_banknotes = pd.read_csv("people-on-banknotes.csv")
print(f"pro Person:   {issued_before_death_percentage(df):.2f} %")
print(f"pro Banknote: {issued_before_death_percentage(all_banknotes):.2f} %")

pro Person:   4.56 %
pro Banknote: 12.19 %


**Antwort:** Nach der Bereinigung gibt es pro Person nur noch eine Zeile. Das betrifft 4,56 % der Personen (11 von 241: 9 mit negativem Wert, 2 mit NaN). Zählt man alle Banknoten im ungekürzten Datensatz, sind es 12,19 % (34 von 279).

Laut Angabe heißt NaN, dass die Person noch lebte. Bei Queen Elizabeth II stimmt das für den Stand des Datensatzes. Bei Genoveva Ríos (Bolivien) fehlt vermutlich nur das Todesjahr, das müsste man noch prüfen. Ohne sie wären es 10 von 240 Personen (4,17 %).

### Who is the oldest historical figure in the dataset?

In [9]:
def oldest_figure(data: pd.DataFrame) -> pd.Series:
    """Liefert die Zeile der Person mit dem frühesten Todesjahr (NaN wird ignoriert).

    >>> toy = pd.DataFrame({"name": ["A", "B", "C"], "death": [1900, -100, None]})
    >>> oldest_figure(toy)["name"]
    'B'
    """
    return data.loc[data["death"].idxmin()]


oldest_figure(df)[["name", "country", "occupation", "death", "first_appearance"]]

name                Hannibal
country              Tunisia
occupation          Military
death                 -183.0
first_appearance        2013
Name: 234, dtype: object

**Antwort:** Hannibal (Tunesien) mit dem Todesjahr -183, also 183 v. Chr. "Älteste Person" ist hier als frühestes Todesjahr gemeint, weil der Datensatz keine Geburtsjahre enthält.